# Lecture 5 — Fit a count regression two ways, and judge the model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/AdvancedStatisticsCourse_36725561/blob/main/notebooks/lecture5_bike_counts.ipynb)


In the lecture, a generalized linear model (GLM) let the linear predictor $\eta=w^\top x$ set the natural parameter of a non-normal response. The mean $\mu=A'(\eta)$ and the variance $\mathrm{Var}(Y\mid x)=A''(\eta)$ gave the gradient and the Hessian of the fitting objective, and two algorithms fit it: Newton's method (IRLS), which uses the Hessian, and minibatch stochastic gradient descent (SGD), which uses only the gradient. Here we fit a **Poisson regression to hourly bike rentals** with both.

**How many rentals should we expect in an hour, how should we compute the fit, how does each algorithm's cost change as the model grows, and does a Poisson model describe these counts?**

| Section | What we do | Contribution to the question |
|---|---|---|
| 1. IRLS | Write the Newton/IRLS update from the lecture's gradient and Hessian; check the fit and plot it | The exact fit, the rentals it predicts for an hour, and whether a Poisson model describes the counts |
| 2. Curvature | The Hessian's largest and smallest eigenvalues | What limits SGD's step and speed |
| 3. SGD | Set the step from the curvature; a larger step diverges; progress is slow | How to compute the fit with the second algorithm, and why it is slow here |
| 4. Adam | A variant of SGD that adapts the step for each coefficient | A faster way to compute the fit |
| 5. Held-out scores | L04's log score against a baseline | How well each fit predicts new hours, and whether the algorithms differ |
| 6. Scaling | Time both algorithms as the number of coefficients grows | How each algorithm's cost changes as the model grows |

PyTorch is new to this course; Section 3 explains the few pieces we use.

### *Explanation to read*

#### The data

The data are the hourly rentals of a bike-sharing system over two years (OpenML `Bike_Sharing_Demand`, version 2, which OpenML attributes to the Capital Bikeshare system, Washington D.C., 2011-2012, with weather and calendar information added): 17,379 hours, $y_n$ the number of rentals in hour $n$. A random 80% (the *training hours*, $N=13{,}903$) are used to fit; the other 20% are *held-out hours* for scoring.

The input $x_n$ has $D=57$ entries: a constant 1; temperature, "feels like" temperature, humidity and wind speed, each standardized with the training hours' mean and standard deviation; 0/1 indicators for season (3 indicators) and weather (2 indicators), each leaving one reference level out (weather has three levels, clear, misty and rain, with the hours of heavy rain counted as rain, as in the lecture); and 47 indicators for the 48 combinations of hour of day and day type, one combination being the reference level. The combination `slot=k` has code $k=2\times\text{hour}+c$, with $c=0$ on a working day and $c=1$ on a day off (the data set's coding), so `slot=9` is hour 4 on a day off. In all, $D=1+4+3+2+47=57$. $X$ is the $N\times D$ matrix with rows $x_n^\top$. The model is our choice, not something the data set prescribes.

### *Code to run*

#### Load the data and build $X$

Loads the hours, makes the split (seed 2027) and builds the design matrix. It prints the sizes and the mean and variance of the counts. `fetch_openml` downloads the data, so the first run needs an internet connection.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.special import gammaln
from sklearn.datasets import fetch_openml
from sklearn.linear_model import PoissonRegressor

torch.set_default_dtype(torch.float64)
plt.rcParams["figure.dpi"] = 110

bike = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=False, parser="liac-arff")
raw = dict(zip(bike.feature_names, bike.data.T))      # column name -> array of 17,379 values
counts = bike.target.astype(float)                    # y: rentals in the hour

rng = np.random.default_rng(2027)
order = rng.permutation(len(counts))
n_train = int(0.8 * len(counts))
train, held = order[:n_train], order[n_train:]


def dummies(codes, labels=None):
    # One 0/1 column per level of a categorical code, leaving out the first level.
    codes = np.asarray(codes).astype(int)
    levels = np.unique(codes)
    names = [labels[v] if labels else int(v) for v in levels[1:]]
    return (codes[:, None] == levels[None, 1:]).astype(float), names


def design(cells):
    # Columns: constant, 4 standardized continuous measures, season, weather, and one
    # indicator per level of `cells` (an integer code for a combination of hour, day type, ...).
    cont_names = ["temp", "feel_temp", "humidity", "windspeed"]
    cont = np.column_stack([raw[k] for k in cont_names])
    cont = (cont - cont[train].mean(0)) / cont[train].std(0)        # training-hour statistics
    season, season_lv = dummies(raw["season"], bike.categories["season"])
    weather_code = np.where(raw["weather"].astype(int) == 1, 3, raw["weather"].astype(int))   # heavy rain (1) counted as rain (3)
    weather, weather_lv = dummies(weather_code, bike.categories["weather"])
    cell, cell_lv = dummies(cells)
    X = np.column_stack([np.ones(len(cont)), cont, season, weather, cell])
    names = (["constant"] + cont_names + [f"season={v}" for v in season_lv]
             + [f"weather={v}" for v in weather_lv] + [f"slot={v}" for v in cell_lv])
    return X, names


hour = raw["hour"].astype(int)
working = raw["workingday"].astype(int)
X_all, names = design(hour * 2 + working)        # hour of day x working day: 48 combinations
X_train, y_train = X_all[train], counts[train]
X_held, y_held = X_all[held], counts[held]
N, D = X_train.shape

print(f"training hours N = {N}, held-out hours = {len(held)}, coefficients D = {D}")
print(f"rentals per hour: mean {y_train.mean():.1f}, variance {y_train.var():.0f}, "
      f"variance / mean = {y_train.var() / y_train.mean():.0f}")

### *Explanation to read*

#### The Poisson GLM and its objective

Given $x_n$ and $w$, the count is $Y_n\mid x_n,w\sim\mathrm{Poisson}(\mu_n)$ with log link: $\eta_n=w^\top x_n$ and $\mu_n=e^{\eta_n}$. For a Poisson response $A(\eta)=e^\eta$, so the mean and the variance are both $\mu_n=A'(\eta_n)=A''(\eta_n)$. We fit $w$ by minimizing $F$, the lecture's log score per hour $S(w)$ on the training hours plus a Gaussian-prior penalty of strength $\rho$,

$$
F(w)=\frac1N\sum_{n=1}^N\bigl(\mu_n-y_n\eta_n+\ln y_n!\bigr)+\frac{\rho}{2}\lVert w\rVert^2 .\tag{1}
$$

Its gradient and Hessian are the lecture's $g$ and $H$ divided by $N$, plus $\rho w$ and $\rho I$, with the variance $v_n=\mu_n$ in the Hessian:

$$
\nabla F=\frac1N\sum_{n}(\mu_n-y_n)\,x_n+\rho w=\frac1N X^\top(\mu-y)+\rho w,\tag{2}
$$

$$
\nabla^2F=\frac1N\sum_{n}\mu_n\,x_nx_n^\top+\rho I=\frac1N X^\top\mathrm{diag}(\mu)\,X+\rho I.\tag{3}
$$

We use $\rho=10^{-4}$ on every coefficient, the constant included, which is a Gaussian prior with variance $\tau^2=1/(N\rho)\approx0.7$. It makes $F$ strictly convex, so one fit exists and is unique. We start both algorithms from the constant-only fit $w^{(0)}=(\ln\bar y,0,\dots,0)$ (the Python name is `w0`), which predicts the mean count $\bar y$ for every hour.

To compare algorithms by one number we use the **gap**, $F(w)-F(w^\star)$, what the lecture calls "above the minimum", with $w^\star$ the minimizer, which IRLS finds in Section 1 (`w_irls`). We call a fit *close enough* when its gap is below the **tolerance** 0.05 nats per hour, and we use this one tolerance for every algorithm below. The lecture used 0.001, but plain SGD at a constant step cannot reach that in the time of a notebook.

### *Code to run*

#### The objective and the per-hour log score

`log_scores_poisson(w, X, y)` returns $-\ln p(y_n\mid x_n,w)=\mu_n-y_n\eta_n+\ln y_n!$ for each row, an array with one entry per hour. `objective` is (1). `w0` is the constant-only fit. `tolerance` is the gap below which a fit is close enough.

In [ ]:
rho = 1e-4
tolerance = 0.05


def log_scores_poisson(w, X, y):
    # -ln p(y_n | x_n, w) for each hour; gammaln(y + 1) is ln y!.
    eta = X @ w
    return np.exp(eta) - y * eta + gammaln(y + 1)


def objective(w, X, y):
    return np.mean(log_scores_poisson(w, X, y)) + 0.5 * rho * np.sum(w ** 2)


w0 = np.zeros(D)
w0[0] = np.log(y_train.mean())
print(f"objective at the constant-only fit w0: {objective(w0, X_train, y_train):.4f}")

## 1. IRLS: the Newton step

### *Explanation to read*

#### Newton's method as a reweighted least-squares fit

Newton's method replaces $w$ by

$$
w\leftarrow w-\bigl(\nabla^2F\bigr)^{-1}\nabla F .\tag{4}
$$

For a GLM this step is the solution of a weighted least-squares problem whose weights are the variances $\mu_n$, hence *iteratively reweighted least squares* (IRLS): the same step as the lecture's, there written as a regression of the working response. In code, (4) is one call to `np.linalg.solve(hess, grad)` rather than an inverse. The Hessian (3) needs $\frac1N\sum_n\mu_nx_nx_n^\top$: in NumPy, `X * mu[:, None]` multiplies row $n$ of `X` by $\mu_n$, so `(X * mu[:, None]).T @ X / N` is the sum.

### *Questions to answer*

#### 1. IRLS fit — code

Write the IRLS loop on the training hours, starting from `w0`. At each iteration compute $\mu$, the gradient (2), the Hessian (3) and the step (4), update $w$, and print the iteration number, the objective (1) and the largest absolute entry of the step. Stop when that is below $10^{-8}$ (allow at most 30 iterations); we run IRLS to convergence because the other algorithms' gaps are measured against its result. Store the final coefficients as `w_irls`. (Check: the objective falls from 86.8 at `w0` to about 15 within the first few iterations, then stops changing in the printed digits, while the largest step keeps shrinking by orders of magnitude; the next cell tests the answer.)

In [ ]:
# answer here

### *Explanation to read*

#### Is the variance equal to the mean?

The Poisson model says $\mathrm{Var}(Y_n\mid x_n)=\mu_n$. The **Pearson dispersion**, $\varphi=\frac1{N-D}\sum_n(y_n-\mu_n)^2/\mu_n$, measures how far the training counts are from that: each term is a squared error divided by the variance the model assigns, and $\varphi$ is about 1 when the variance really equals the mean. A value of 25 would mean squared errors about 25 times what the Poisson model allows.

### *Code to run*

#### Check the IRLS fit

Two checks, then two quantities. At the minimum the gradient (2) is zero: the printed largest component should be at rounding-error size, about $10^{-12}$ or smaller. scikit-learn's `PoissonRegressor` with `alpha` equal to $\rho$ minimizes the same objective (up to a constant); we leave its intercept off because our constant column is already in $X$. The largest coefficient difference should be about $10^{-4}$ or smaller, because scikit-learn stops at its own tolerance. The cell also prints $\varphi$ and stores `f_star`, the objective at the fit.

In [ ]:
mu = np.exp(X_train @ w_irls)
grad = X_train.T @ (mu - y_train) / N + rho * w_irls
print("largest |gradient| component at the IRLS fit:", np.max(np.abs(grad)))

sk = PoissonRegressor(alpha=rho, fit_intercept=False, tol=1e-10, max_iter=10000).fit(X_train, y_train)
print("largest |w_IRLS - w_scikit-learn|:", np.max(np.abs(w_irls - sk.coef_)))

dispersion = np.sum((y_train - mu) ** 2 / mu) / (N - D)
print(f"Pearson dispersion {dispersion:.1f}")

f_star = objective(w_irls, X_train, y_train)
print(f"objective at the IRLS fit, f_star = {f_star:.6f}")

### *Code to run*

#### Plot the fit against the training counts

Two panels, on the training hours. Left: the mean rentals in each hour of the day, for working days and for days off: the observed mean (dots) and the mean of the fitted $\mu_n$ over the same hours (lines). Right: the same two means for ten groups of hours with equal numbers of hours, ordered by temperature, plotted against each group's mean temperature (°C). The cell also prints the largest relative difference between the fitted and the observed mean over the 48 hour-and-day groups and over the 10 temperature groups.

In [ ]:
mu_train = np.exp(X_train @ w_irls)
hour_train, working_train, temp_train = hour[train], working[train], raw["temp"][train]
day_names = ["working day" if v == "True" else "day off" for v in bike.categories["workingday"]]   # codes 0 and 1

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
hour_gap = 0.0
for c, colour in [(0, "C0"), (1, "C1")]:
    obs = np.array([y_train[(hour_train == h) & (working_train == c)].mean() for h in range(24)])
    fit = np.array([mu_train[(hour_train == h) & (working_train == c)].mean() for h in range(24)])
    axes[0].plot(range(24), obs, colour + "o", ms=4, label=f"{day_names[c]}: observed mean")
    axes[0].plot(range(24), fit, colour + "-", label=f"{day_names[c]}: fitted mean")
    hour_gap = max(hour_gap, np.max(np.abs(fit - obs) / obs))
axes[0].set_xlabel("hour of day"); axes[0].set_ylabel("mean rentals per hour")
axes[0].set_title("By hour and day type"); axes[0].legend(fontsize=7)

edges = np.quantile(temp_train, np.linspace(0, 1, 11))
group = np.digitize(temp_train, edges[1:-1])                    # 0, ..., 9
temp_mean = np.array([temp_train[group == g].mean() for g in range(10)])
obs = np.array([y_train[group == g].mean() for g in range(10)])
fit = np.array([mu_train[group == g].mean() for g in range(10)])
axes[1].plot(temp_mean, obs, "ko", ms=4, label="observed mean")
axes[1].plot(temp_mean, fit, "k-", label="fitted mean")
axes[1].set_xlabel("temperature (°C), mean of each tenth of the hours"); axes[1].set_ylabel("mean rentals per hour")
axes[1].set_title("By temperature"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"largest |fitted - observed| / observed: hour-and-day groups {hour_gap:.4f}, "
      f"temperature groups {np.max(np.abs(fit - obs) / obs):.3f}")

### *Questions to answer*

#### 2. Why the means agree by hour but not by temperature — written answer

By hour and day type, the fitted and observed means agree in every group; by temperature they do not. The gradient (2) is zero at the fit, and the column of $X$ for the indicator of one hour-and-day group is 1 for the hours in that group and 0 for the others. Use this to explain why the fitted and the observed means must agree (up to the small penalty) in each hour-and-day group, and why the same argument does not make them agree in each temperature group, where the model has the single column `temp` (and `feel_temp`). Then say what the right panel shows about how well the model fits.

- answer here

#### 3. The rentals expected in a misty November morning — code

The lecture's data table includes a misty November morning in 2011: 08:00, 22.1 °C, humidity 68 %, wind speed 7.0. It is row 7529 of `X_all` (a training hour; its count is `counts[7529]`). Compute the expected number of rentals $\mu=e^{w^\top x}$ for that hour from `w_irls` and print it beside the count observed in that hour. In a comment, say how the two compare.

In [ ]:
# answer here

### *Explanation to read*

#### Section summary

IRLS reaches the minimum of (1) in nine iterations, each a weighted least-squares solve; after four the objective is within 0.02 of its final value. The fitted mean equals the observed mean in every hour-and-day group, as the indicators' gradient entries require, but not by temperature, where it is up to 26% off. For the lecture's misty November morning the fit expects about 629 rentals and 480 were observed. The Pearson dispersion is 24.7: the squared errors are about 25 times the Poisson variance.

## 2. Curvature: what limits gradient descent

### *Explanation to read*

#### The Hessian's eigenvalues

Near the minimum, $F$ is close to a quadratic with the Hessian $H=\nabla^2F(w_{\rm IRLS})$ (3). Write $H=\sum_j\lambda_ju_ju_j^\top$, with eigenvalues $\lambda_j>0$ (the curvatures along the principal directions $u_j$). A gradient step of size $\epsilon$ multiplies the remaining error along $u_j$ by

$$
1-\epsilon\lambda_j ,\tag{5}
$$

as in the lecture's one-dimensional quadratic. Every direction converges only if $\epsilon\lambda_j<2$ for all $j$, that is, $\epsilon<2/\lambda_{\max}$, and a direction with small curvature converges slowly. Minibatch gradients are noisy, so we take $\epsilon=1/\lambda_{\max}$, half the limit. `np.linalg.eigh(H)` returns the eigenvalues of a symmetric matrix in increasing order, and the matching eigenvectors as columns ([NumPy documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html)).

### *Questions to answer*

#### 4. The curvature and the step — code

Form $H$ at `w_irls` and find its eigenvalues with `np.linalg.eigh`. Store the eigenvalues and eigenvectors as `eigenvalues` and `eigenvectors`, the extreme ones as `lam_min` and `lam_max`, the ratio `kappa = lam_max / lam_min`, and `step_sgd = 1 / lam_max`. Print `lam_max`, `lam_min`, `kappa` and `step_sgd`. (Check: the largest eigenvalue is a few hundred, and the ratio is of order $10^4$.)

In [ ]:
# answer here

### *Code to run*

#### Which direction is flattest?

For the flattest, second flattest and steepest directions, prints the four coefficients with the largest entries in that direction's eigenvector (the entry is in parentheses).

In [ ]:
for label, k in [("flattest", 0), ("second flattest", 1), ("steepest", -1)]:
    v = eigenvectors[:, k]
    top = ", ".join(f"{names[j]} ({v[j]:+.2f})" for j in np.argsort(-np.abs(v))[:4])
    print(f"{label:<16s} (curvature {eigenvalues[k]:8.4f}): {top}")

### *Explanation to read*

#### Section summary

The curvatures range from about 0.021 to 499, so $\kappa=\lambda_{\max}/\lambda_{\min}\approx2.4\times10^4$. The flattest direction mixes the early-morning hour-and-day indicators (hours 1 to 6), whose hours have few rentals (5 to 8 on average for hours 3 to 5, against 190 overall), so the weights $\mu_n$ in (3) are small; the second flattest is mostly `slot=9`, hour 4 on a day off. The steepest combine the constant with the correlated temperature measures.

## 3. SGD with a step set from the curvature

### *Explanation to read*

#### PyTorch in five lines

We run SGD in PyTorch, which computes gradients automatically. You need only these pieces ([autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)):

- A *tensor* is NumPy's array with the same arithmetic (`@`, `*`, `torch.exp`, `torch.mean`); `torch.from_numpy(a)` converts an array, `.detach().numpy()` converts back.
- A tensor created with `requires_grad=True`, such as the coefficients `w`, records the operations applied to it. `loss.backward()` then fills `w.grad` with $\partial\,\text{loss}/\partial w$ (automatic differentiation, covered in recording L05-R01).
- An *optimizer*, such as `torch.optim.SGD([w], lr=step)`, holds `w`. `opt.step()` updates `w` from `w.grad` (for SGD, $w\leftarrow w-\epsilon\,\texttt{w.grad}$); `opt.zero_grad()` clears old gradients first.

The next cell checks that PyTorch's gradient is our formula (2), at a point away from the minimum.

### *Code to run*

#### PyTorch's gradient equals the formula (2)

Computes the loss with `requires_grad=True` coefficients and compares `w.grad` with the formula.

In [ ]:
X_t, y_t = torch.from_numpy(X_train), torch.from_numpy(y_train)

w_test = w_irls + 0.1 * np.random.default_rng(0).standard_normal(D)
w_torch = torch.tensor(w_test, requires_grad=True)
eta = X_t @ w_torch
loss = torch.mean(torch.exp(eta) - y_t * eta) + 0.5 * rho * torch.sum(w_torch ** 2)    # (1) without ln y!, which has no gradient
loss.backward()

mu_test = np.exp(X_train @ w_test)
formula = X_train.T @ (mu_test - y_train) / N + rho * w_test
print("largest |autograd gradient - formula gradient|:", np.max(np.abs(w_torch.grad.numpy() - formula)))

### *Explanation to read*

#### The minibatch loop

`fit_minibatch` below runs minibatch gradient steps from `w0`. An *epoch* shuffles the training hours and goes through them in batches of 256 (55 steps). A step computes the batch average of (1), whose gradient is an unbiased estimate of (2), calls `backward()`, and lets the optimizer update `w`. After each epoch it records the gap: the full-data objective (1) minus `f_star`. Its arguments are `optimizer` (`"sgd"` or `"adam"`), `step` $\epsilon$, `epochs`, and `decay_steps` (`None` for a constant step; Section 4 uses it). It returns the gaps and the coefficients after every epoch, and stops early if the gap is not finite.

### *Code to run*

#### The minibatch loop

Defines `fit_minibatch`, which returns the gap after each epoch.

In [ ]:
steps_per_epoch = int(np.ceil(N / 256))


def fit_minibatch(optimizer, step, epochs, decay_steps=None, batch=256, seed=0):
    # Minibatch gradient steps from w0; returns the gap after each epoch and the coefficients after each epoch.
    w = torch.tensor(w0.copy(), requires_grad=True)
    opt = torch.optim.SGD([w], lr=step) if optimizer == "sgd" else torch.optim.Adam([w], lr=step)
    if decay_steps:      # the step shrinks like 1 / (1 + t / decay_steps)
        schedule = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 1 / (1 + t / decay_steps))
    generator = torch.Generator().manual_seed(seed)
    gaps, path = [], []
    for epoch in range(epochs):
        shuffled = torch.randperm(N, generator=generator)
        for start in range(0, N, batch):
            idx = shuffled[start:start + batch]
            eta = X_t[idx] @ w
            loss = torch.mean(torch.exp(eta) - y_t[idx] * eta) + 0.5 * rho * torch.sum(w ** 2)
            opt.zero_grad()
            loss.backward()
            opt.step()
            if decay_steps:
                schedule.step()
        w_now = w.detach().numpy().copy()
        gap = objective(w_now, X_train, y_train) - f_star
        gaps.append(gap)
        path.append(w_now)
        if not np.isfinite(gap):
            break
    return np.array(gaps), np.array(path)

### *Questions to answer*

#### 5. A safe step and a step ten times larger — code

Run `fit_minibatch("sgd", ...)` twice: with step `step_sgd` for 300 epochs (15 seconds to a minute, depending on the machine), storing the result as `gaps_sgd, path_sgd`; and with step `10 * step_sgd` for 5 epochs, storing `gaps_big, _`. Print the first gap of the 300-epoch run, its gaps after 10, 100 and 300 epochs, and all the gaps of the larger step. In a comment, say what happened with the larger step and how it relates to the limit $2/\lambda_{\max}$ from (5).

In [ ]:
# answer here

### *Explanation to read*

#### Where the gap comes from

Near the minimum $F$ is close to its quadratic model, so the gap is $\tfrac12\sum_j\lambda_je_j^2$, where $e_j=u_j^\top(w-w_{\rm IRLS})$ is the remaining error along direction $j$. Direction $j$ therefore *contributes* $\tfrac12\lambda_je_j^2$ to the gap. A flat direction can hold a large error and still contribute little, and a steep direction with a small error can contribute as much. The next cell plots the errors along the two flattest directions and the gap, and prints the three largest contributions after 300 epochs.

### *Code to run*

#### Errors along the flattest directions

Left: the observed error of the SGD path along each of the two flattest eigenvectors, $u_j^\top(w-w_{\rm IRLS})$ after each epoch (solid), against the prediction from (5) after $t=55\times\text{epoch}$ steps, $(1-\epsilon\lambda_j)^t$ times the starting error (dashed). Right: the gap of Question 5 against the tolerance. Below the plots, the actual gap and its quadratic approximation, then the three largest contributions $\tfrac12\lambda_je_j^2$. It uses `eigenvalues`, `eigenvectors`, `path_sgd` and `gaps_sgd` from your answers.

In [ ]:
epochs = np.arange(1, 301)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for j, colour in [(0, "C0"), (1, "C1")]:
    e0 = eigenvectors[:, j] @ (w0 - w_irls)
    observed = path_sgd @ eigenvectors[:, j] - eigenvectors[:, j] @ w_irls
    predicted = e0 * (1 - step_sgd * eigenvalues[j]) ** (epochs * steps_per_epoch)
    axes[0].plot(epochs, np.abs(observed), colour, label=f"direction {j + 1} (curvature {eigenvalues[j]:.3f}): SGD")
    axes[0].plot(epochs, np.abs(predicted), colour + "--", label="prediction $(1-\\epsilon\\lambda_j)^t$")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("|error along direction|"); axes[0].set_yscale("log")
axes[0].set_title("The two flattest directions"); axes[0].legend(fontsize=7)

axes[1].plot(epochs, gaps_sgd, label="SGD, step $1/\\lambda_{\\max}$")
axes[1].axhline(tolerance, color="k", lw=0.8, ls=":", label="tolerance")
axes[1].set_yscale("log"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("training objective minus IRLS optimum")
axes[1].set_title("Gap to the IRLS fit"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Where the gap after 300 epochs comes from: the quadratic approximation sum_j (1/2) lambda_j e_j^2.
e_final = eigenvectors.T @ (path_sgd[-1] - w_irls)
share = 0.5 * eigenvalues * e_final ** 2
print(f"gap after 300 epochs {gaps_sgd[-1]:.3f}; quadratic approximation {share.sum():.3f}")
for j in np.argsort(-share)[:3]:
    print(f"  curvature {eigenvalues[j]:8.3f}: error {e_final[j]:+.3f}, contribution {share[j]:.3f}")

### *Questions to answer*

#### 6. Steps and epochs per factor of $e$ — written answer

At step $\epsilon=1/\lambda_{\max}$, (5) says the error along the flattest direction shrinks by $1-1/\kappa$ per step. Using $(1-1/\kappa)^t\approx e^{-t/\kappa}$, about how many steps, and how many epochs (55 steps each), does it take to shrink by a factor of $e$?

- answer here

#### 7. Does the flattest direction fall at that rate? — written answer

In the left panel, does the solid curve of direction 1 fall at about the rate of Question 6, as the dashed prediction does?

- answer here

#### 8. What keeps the gap above the tolerance — written answer

Using the printed contributions, what keeps the gap in the right panel above the dotted tolerance after 300 epochs?

- answer here

### *Explanation to read*

#### Section summary

A step of $1/\lambda_{\max}$ is stable and ten times that diverges, as (5) predicts. At the stable step the error along the flattest direction falls at about the rate (5) gives, so with $\kappa\approx2.4\times10^4$ it takes hundreds of epochs per factor of $e$, and the steepest direction's minibatch noise is not removed by a constant step. After 300 epochs the gap, 0.19, is still above the tolerance, with about equal parts from the slow flat direction and from the noise.

## 4. Adam: a variant of SGD

### *Explanation to read*

#### A step adapted to each coefficient

**Adam** is a variant of SGD that adapts the step separately for each coefficient: it divides each coefficient's update by a running estimate of the size of that coefficient's recent gradients ([PyTorch documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)). It is the default optimizer in most PyTorch code. We run it with step 0.3 that shrinks like $1/(1+t/100)$ (a shrinking step lets the noise settle). These settings were tuned for this problem and are not derived, and Adam's speed depends on them.

### *Code to run*

#### Adam on the same objective

Runs 100 epochs of Adam with the same batches and starting point, stores `gaps_adam` and `path_adam`, prints the first epoch whose gap is below the tolerance, and plots both gaps.

In [ ]:
gaps_adam, path_adam = fit_minibatch("adam", 0.3, epochs=100, decay_steps=100)

adam_epochs = int(np.nonzero(gaps_adam < tolerance)[0][0]) + 1
print(f"Adam: first epoch with gap below {tolerance}: {adam_epochs}; gap after 100 epochs {gaps_adam[-1]:.3f}")
print(f"plain SGD after 300 epochs: gap {gaps_sgd[-1]:.3f}")

plt.figure(figsize=(6, 3.8))
plt.plot(np.arange(1, 301), gaps_sgd, label="SGD, step $1/\\lambda_{\\max}$")
plt.plot(np.arange(1, 101), gaps_adam, label="Adam, decaying step")
plt.axhline(tolerance, color="k", lw=0.8, ls=":", label="tolerance")
plt.yscale("log"); plt.xlabel("epoch (pass through the training hours)")
plt.ylabel("training objective minus IRLS optimum"); plt.legend(); plt.tight_layout(); plt.show()

### *Questions to answer*

#### 9. A hypothesis for why Adam is faster here — written answer

Adam reaches the tolerance within 100 epochs; plain SGD does not within 300. Using Section 2's finding about which coefficients make up the flat directions, and what Adam adapts, state a hypothesis for why Adam is faster here, and describe one experiment that would test it.

- answer here

### *Explanation to read*

#### Section summary

Adam reaches a gap below 0.05 in 59 epochs; plain SGD at step $1/\lambda_{\max}$ is still above it after 300 (gap 0.19). Both fit the same convex objective and differ in how the step is chosen.

## 5. Held-out scores

### *Explanation to read*

#### Scoring the fits on hours they did not see

As in L04, we score a fit by its **held-out log score**, the average of $-\ln p(y_m\mid x_m,w)$ over the $M$ held-out hours (`len(held)`); lower is better. The training hours would flatter the fit, because $w$ was chosen to score well on them. To compare fits $A$ and $B$ on the same hours, take the per-hour difference $d_m$ = (score of $A$ on hour $m$) $-$ (score of $B$ on hour $m$). Its mean $\bar d$ has standard error $\mathrm{sd}(d)/\sqrt M$, with `np.std(d, ddof=1)`, and a $\bar d$ more than about two standard errors from 0 is unlikely to be chance alone. The baseline ignores the covariates: every hour gets the same Poisson distribution, with the training hours' mean rate, which is the constant-only fit `w0`. `log_scores_poisson(w, X, y)` gives the per-hour scores of a coefficient vector on the hours in `X`, `y`.

### *Questions to answer*

#### 10. Training and held-out log scores — code

Make a dictionary `fits` of coefficient vectors with the keys "baseline: constant rate" (`w0`), "Poisson GLM, IRLS" (`w_irls`), "Poisson GLM, plain SGD (300 epochs)" (`path_sgd[-1]`) and "Poisson GLM, Adam (100 epochs)" (`path_adam[-1]`). Compute each fit's per-hour scores on the training hours (`X_train`, `y_train`) and on the held-out hours (`X_held`, `y_held`); store the held-out scores in a dictionary `scores` with the same keys and `M = len(held)`, and print each fit's two means side by side, in nats per hour. In a comment, say how each fit's training score compares with its held-out score.

In [ ]:
# answer here

#### 11. Paired held-out differences — code

Using `scores`, print the mean and standard error of the paired held-out differences baseline minus IRLS, plain SGD minus IRLS and Adam minus IRLS. In a comment, say how much the covariates help and whether either SGD fit differs from IRLS by more than two standard errors.

In [ ]:
# answer here

### *Explanation to read*

#### Section summary

The covariates are worth about 69 nats per hour over the constant rate. Adam scores within two standard errors of IRLS (0.021 $\pm$ 0.013); plain SGD scores about 0.12 worse (2.1 standard errors), which is small next to 69. Each fit scores about the same on the training hours as on held-out hours: with 57 coefficients and 13,903 hours, the fits do not overfit.

## 6. Scaling: when does SGD overtake IRLS?

### *Explanation to read*

#### The cost of one step

One IRLS iteration forms $X^\top\mathrm{diag}(\mu)X$ and solves with it, about $ND^2+D^3$ operations, and needs few iterations. One SGD epoch costs about $ND$, and many epochs are needed. So IRLS should become the more expensive algorithm as the number of coefficients $D$ grows. The next cell tests this: it adds indicators (hour by weekday; by season; by month) to reach $D$ of 57, 177, 681 and 2,025, and times both algorithms to the same gap, the tolerance of Section 1. Adam (Section 4's settings) is stopped at its first epoch with a gap below the tolerance. IRLS is run to convergence to find its optimum, and its time is the time to the first iteration whose gap is below the tolerance; the evaluation of the objective is not timed for either algorithm. For the largest model it times one IRLS iteration and one Adam epoch (the shortest of three repeats) and multiplies by the typical counts. It takes from 15 seconds to a few minutes, depending on the machine and what else it is running; times on a hosted runtime will differ.

### *Code to run*

#### Time both algorithms as $D$ grows

Prints one row per model size (the iterations and epochs are those needed to reach the tolerance) and plots the times on log axes. Open markers are implied, not measured, totals.

In [ ]:
def time_irls(X, y, one_iteration=False):
    # Newton iterations from the constant-only fit. Returns the final coefficients, the objective after each iteration
    # and the cumulative time after each iteration; evaluating the objective is not timed.
    Nn, Dd = X.shape
    w = np.zeros(Dd)
    w[0] = np.log(y.mean())
    elapsed, objectives, times = 0.0, [], []
    for it in range(1, 40):
        t0 = time.perf_counter()
        mu = np.exp(X @ w)
        grad = X.T @ (mu - y) / Nn + rho * w
        hess = (X * mu[:, None]).T @ X / Nn + rho * np.eye(Dd)
        step = np.linalg.solve(hess, grad)
        w = w - step
        elapsed += time.perf_counter() - t0
        objectives.append(objective(w, X, y))
        times.append(elapsed)
        if one_iteration or np.max(np.abs(step)) < 1e-8:
            break
    return w, np.array(objectives), np.array(times)


def irls_to_tolerance(X, y):
    # Iterations and time of IRLS up to its first iteration with gap below the tolerance, and its optimum.
    w, objectives, times = time_irls(X, y)
    first = int(np.nonzero(objectives - objectives[-1] < tolerance)[0][0])
    return w, objectives[-1], first + 1, times[first]


def time_adam(X, y, f_target, max_epochs=150, one_epoch=False):
    Nn, Dd = X.shape
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    w = torch.zeros(Dd)
    w[0] = float(np.log(y.mean()))
    w.requires_grad_(True)
    opt = torch.optim.Adam([w], lr=0.3)
    schedule = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: 1 / (1 + t / 100))
    generator = torch.Generator().manual_seed(0)
    training_time = 0.0
    for epoch in range(1, (1 if one_epoch else max_epochs) + 1):
        t0 = time.perf_counter()
        shuffled = torch.randperm(Nn, generator=generator)
        for start in range(0, Nn, 256):
            idx = shuffled[start:start + 256]
            eta = Xt[idx] @ w
            loss = torch.mean(torch.exp(eta) - yt[idx] * eta) + 0.5 * rho * torch.sum(w ** 2)
            opt.zero_grad(); loss.backward(); opt.step(); schedule.step()
        training_time += time.perf_counter() - t0          # the gap check below is not timed
        if not one_epoch and objective(w.detach().numpy(), X, y) - f_target < tolerance:
            break
    return w.detach().numpy(), epoch, training_time


day = raw["weekday"].astype(int)
season_code = raw["season"].astype(int)
month = raw["month"].astype(int)
ladder = [("hour x working day", hour * 2 + working),
          ("hour x weekday", hour * 7 + day),
          ("hour x weekday x season", (hour * 7 + day) * 4 + season_code),
          ("hour x weekday x month", (hour * 7 + day) * 12 + month)]

print(f"{'indicators':<26s}{'D':>6s} | {'IRLS its':>8s}{'seconds':>9s} | {'Adam epochs':>11s}{'seconds':>9s}")
rows = []
for label, cells_ in ladder[:3]:
    X_a, _ = design(cells_)
    Xa_train = X_a[train]
    w_i, f_opt, its, t_i = irls_to_tolerance(Xa_train, y_train)
    w_a, eps, t_a = time_adam(Xa_train, y_train, f_opt)
    rows.append((Xa_train.shape[1], its, t_i, eps, t_a))
    print(f"{label:<26s}{Xa_train.shape[1]:>6d} | {its:>8d}{t_i:>9.2f} | {eps:>11d}{t_a:>9.2f}")

# Largest model: one IRLS iteration and one Adam epoch, then the totals implied by the counts above.
label, cells_ = ladder[3]
X_a, _ = design(cells_)
Xa_train = X_a[train]
# the shortest of three repeats, because other programs running on the machine only ever add time
t_one_irls = min(time_irls(Xa_train, y_train, one_iteration=True)[2][0] for _ in range(3))
t_one_epoch = min(time_adam(Xa_train, y_train, None, one_epoch=True)[2] for _ in range(3))
its_typ = int(np.median([r[1] for r in rows])); eps_typ = int(np.median([r[3] for r in rows]))
print(f"{label:<26s}{Xa_train.shape[1]:>6d} | one IRLS iteration {t_one_irls:.2f} s, one Adam epoch {t_one_epoch:.2f} s;"
      f" implied totals: IRLS {its_typ * t_one_irls:.0f} s, Adam {eps_typ * t_one_epoch:.0f} s (not measured end to end)")

D_list = [r[0] for r in rows] + [Xa_train.shape[1]]
irls_total = [r[2] for r in rows] + [its_typ * t_one_irls]
adam_total = [r[4] for r in rows] + [eps_typ * t_one_epoch]
plt.figure(figsize=(6, 3.8))
plt.loglog(D_list, irls_total, "o-", label="IRLS to the tolerance")
plt.loglog(D_list, adam_total, "s-", label="Adam to the tolerance")
plt.loglog(D_list[-1:], irls_total[-1:], "o", mfc="none", ms=10, color="C0")
plt.loglog(D_list[-1:], adam_total[-1:], "s", mfc="none", ms=10, color="C1")
plt.xlabel("number of coefficients D"); plt.ylabel("seconds (open markers: implied, not measured)")
plt.legend(); plt.tight_layout(); plt.show()

### *Questions to answer*

#### 12. Where the curves cross — written answer

From your own table and plot: roughly at what number of coefficients $D$ do the IRLS and Adam curves cross?

- answer here

### *Explanation to read*

#### Section summary

To the same gap, below 0.05, IRLS needs 4 iterations and Adam about 58 epochs at each of the three sizes timed, so the growth with $D$ is in the seconds per iteration or per epoch. An IRLS iteration costs about $ND^2+D^3$, which grows quickly with $D$ (2 to 7 s at $D=2{,}025$), while an Adam epoch costs about $ND$ and grows slowly (0.1 to 0.25 s at $D=2{,}025$). IRLS is faster up to several hundred coefficients and Adam at about 2,000, with the crossover between 700 and 2,000. Where they cross depends on the machine, the accuracy asked for, $N$ and the tuning.

### *Explanation to read*

#### Answer to the opening question

**How many rentals to expect.** For the lecture's misty November morning the fit expects about 629 rentals, against 480 observed. Over held-out hours, the Poisson regression's mean log score is about 15.4 nats per hour, against 84.8 for the constant-rate baseline: hour, day type, season, weather and temperature explain much of the variation in the expected count.

**How to compute the fit.** IRLS reaches the exact minimum in nine iterations, and a gap below the tolerance in four. Plain minibatch SGD needs a step below $2/\lambda_{\max}$; at $1/\lambda_{\max}$ it is still above the tolerance after 300 epochs, because $\kappa\approx2.4\times10^4$ makes the flattest direction slow and the constant step leaves minibatch noise. Adam, which adapts the step for each coefficient, reached the tolerance in 59 epochs and scored like IRLS.

**How the cost changes with the model.** IRLS costs about $ND^2+D^3$ per iteration; an SGD epoch about $ND$. Here IRLS was faster up to about 700 coefficients and Adam at about 2,000, with the crossover between 1,500 and 1,900 on the machine where the notebook was written.

**Whether Poisson describes the counts.** Not entirely. The fitted mean matches the observed mean in every hour-and-day group, but by temperature it is off by up to 26% (the fit plot), and the spread is wrong: the Pearson dispersion is 24.7, so the squared errors are about 25 times the Poisson variance, and the misty morning's count is about 6 Poisson standard deviations from its expectation but about 1 on the scale of the dispersion.